In [ ]:
pip install langchain openai faiss-cpu pypdf tiktoken sentence-transformers langchain-community langchain-text-splitters langchain-huggingface langchain-openai

In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

def load_and_chunk(pdf_path):
    loader = PyPDFLoader(pdf_path)
    documents = loader.load()

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000,
        chunk_overlap=150,
        separators=["\n\n", "\n", ".", " "]
    )

    chunks = splitter.split_documents(documents)
    return chunks

docs = load_and_chunk("/content/Sample-Financial-Statements-1.pdf")
print(f"Total chunks: {len(docs)}")

Total chunks: 3


In [ ]:
from langchain_huggingface.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

vectorstore = FAISS.from_documents(docs, embedding_model)

# Save index (important for reuse)
vectorstore.save_local("faiss_index")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 4}  # reduce tokens → lower cost
)

In [ ]:
from langchain_core.prompts import PromptTemplate

prompt_template = """
You are a financial document assistant.

Answer ONLY using the provided context.
If the answer is not in the context, say:
"I could not find this information in the document."

Context:
{context}

Question:
{question}

Answer:
"""

prompt = PromptTemplate(
    template=prompt_template,
    input_variables=["context", "question"]
)

In [ ]:
import os
from langchain_openai import ChatOpenAI

# Set your OpenAI API key as an environment variable
os.environ["OPENAI_API_KEY"] = ""
#os.environ["OPENAI_API_KEY"] = "sk-proj-xxxxx"
llm = ChatOpenAI(
    model="gpt-4o-mini",  # cost-efficient
    temperature=0
)

In [ ]:
def format_docs(docs):
    return "\n\n".join([doc.page_content for doc in docs])

def rag_pipeline(query):
    retrieved_docs = retriever.get_relevant_documents(query)
    context = format_docs(retrieved_docs)

    final_prompt = prompt.format(
        context=context,
        question=query
    )

    response = llm.predict(final_prompt)
    return response

In [ ]:
from langchain_core.messages import HumanMessage

def rag_pipeline(query):
    retrieved_docs = retriever.get_relevant_documents(query)
    context = format_docs(retrieved_docs)

    final_prompt = prompt.format(
        context=context,
        question=query
    )

    response = llm.invoke([HumanMessage(content=final_prompt)])

    return response.content

In [ ]:
def evaluate_retrieval(query, expected_keywords):
    docs = retriever.invoke(query)
    text = " ".join([d.page_content for d in docs])

    score = sum(1 for kw in expected_keywords if kw.lower() in text.lower())
    return score / len(expected_keywords)

score = evaluate_retrieval(
    "revenue growth risks",
    ["risk", "revenue", "decline"]
)

print("Retrieval Accuracy:", score)

Retrieval Accuracy: 0.3333333333333333
